# Tourenkarte für Soll- und Ist-Daten (EMR alle)

Dieses Notebook liest `IST_DATEN_MEHR_FORMULAR.csv` und `Soll-Daten.csv` ein und erstellt eine interaktive Karte mit:

- Soll- und Ist-Touren
- verbundenen Punkten
- Hover-Infos für Abfahrt und Ankunft
- Stoppreihenfolge als Zahl im Marker
- automatischem Zoom auf die gewählte Tour
- umschaltbaren Tournummern über die Layer-Auswahl

Diese Ist-Daten sind die EMR Daten mit fast allen Formularbeschreibungen (mit Ausnahme von welchen, die sehr selten vorkommen) und zugeordneter Tournummer.


In [1]:
import pandas as pd
import folium
from folium.plugins import BeautifyIcon
from branca.element import Template, MacroElement
from IPython.display import display


In [5]:
IST_FILE = 'IST_DATEN_MEHR_FORMULAR.csv'
SOLL_FILE = 'Soll-Daten.csv'
OUTPUT_HTML = 'tourenkarte_emr_alle.html'


def normalize_tour(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None


def parse_dt(series):
    return pd.to_datetime(series, errors='coerce')


def fmt_dt(x):
    if pd.isna(x):
        return '-'
    return pd.Timestamp(x).strftime('%d.%m.%Y %H:%M')

def clean_ist(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    df["TOURNR"] = df["TOURNR"].astype(str).str.strip()

    df["Breitengrad"] = (
        df["Breitengrad"]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
    )
    df["Längengrad"] = (
        df["Längengrad"]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

    df["Breitengrad"] = pd.to_numeric(df["Breitengrad"], errors="coerce")
    df["Längengrad"] = pd.to_numeric(df["Längengrad"], errors="coerce")

    df["Meldungszeit"] = pd.to_datetime(df["Meldungszeit"], errors="coerce")

    df = df.dropna(subset=["TOURNR", "Breitengrad", "Längengrad"]).copy()

    df = df.sort_values(["TOURNR", "Meldungszeit", "DATUM"]).reset_index(drop=True)
    df["stop_no"] = df.groupby("TOURNR").cumcount() + 1

    return df


def clean_soll(df):
    df = df.copy()
    df['TOURNR'] = df['TOURNR'].map(normalize_tour)
    df['GEOX'] = pd.to_numeric(df['GEOX'], errors='coerce')
    df['GEOY'] = pd.to_numeric(df['GEOY'], errors='coerce')
    df['ABFAHRT_dt'] = parse_dt(df['ABFAHRT'])
    df['ANKUNFT_dt'] = parse_dt(df['ANKUNFT'])

    df = df.dropna(subset=['TOURNR', 'GEOY', 'GEOX'])
    df = df.sort_values(['TOURNR', 'ABFAHRT_dt', 'ANKUNFT_dt', 'DATUM']).reset_index(drop=True)
    df['stop_no'] = df.groupby('TOURNR').cumcount() + 1
    return df


In [6]:
df_ist = pd.read_csv(IST_FILE)
df_soll = pd.read_csv(SOLL_FILE)

df_ist = clean_ist(df_ist)
df_soll = clean_soll(df_soll)

print('Ist-Form:', df_ist.shape)
print('Soll-Form:', df_soll.shape)
print('Anzahl Touren in Ist:', df_ist['TOURNR'].nunique())
print('Anzahl Touren in Soll:', df_soll['TOURNR'].nunique())

display(df_ist.head())
display(df_soll.head())


Ist-Form: (114756, 18)
Soll-Form: (1620, 13)
Anzahl Touren in Ist: 205
Anzahl Touren in Soll: 240


,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR,stop_no
0,8317.0,"Hesse, Sven-Erik",30019,Rollen,2026-03-02 06:15:57,0,4,339687.575,0.0,0.0,53.52326,9.98572,"DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",Plö-TS856 - Fahrzeugdetails,2026-03-02,Plo-TS856,H/42374,1
1,8317.0,"Hesse, Sven-Erik",1644,Beginn Rollen,2026-03-02 06:16:22,129,6,339687.595,0.0,0.0,53.52346,9.98562,"DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",Plö-TS856 - Fahrzeugdetails,2026-03-02,Plo-TS856,H/42374,2
2,8317.0,"Hesse, Sven-Erik",30019,Rollen,2026-03-02 06:16:25,26,5,339687.590,0.0,0.0,53.52348,9.98567,"DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",Plö-TS856 - Fahrzeugdetails,2026-03-02,Plo-TS856,H/42374,3
3,8317.0,"Hesse, Sven-Erik",1643,Ende Rollen,2026-03-02 06:16:35,63,3,339687.615,0.0,0.0,53.52344,9.98594,"DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",Plö-TS856 - Fahrzeugdetails,2026-03-02,Plo-TS856,H/42374,4
4,8317.0,"Hesse, Sven-Erik",1644,Beginn Rollen,2026-03-02 06:16:38,52,6,339687.615,0.0,0.0,53.52346,9.98599,"DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",Plö-TS856 - Fahrzeugdetails,2026-03-02,Plo-TS856,H/42374,5


,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM,ABFAHRT_dt,ANKUNFT_dt,stop_no
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02,2026-03-02 06:30:00,2026-03-02 06:00:00,1
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02,2026-03-02 07:19:00,2026-03-02 06:49:00,2
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02,2026-03-02 08:11:00,2026-03-02 07:41:00,3
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02,2026-03-02 09:01:00,2026-03-02 08:31:00,4
4,H/42375,OD-TS8050,"Szmajdewicz, Marcin",H/42375,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02,2026-03-02 06:30:00,2026-03-02 06:00:00,1


In [7]:
all_tours = sorted(set(df_ist['TOURNR'].dropna()) | set(df_soll['TOURNR'].dropna()))
if not all_tours:
    raise ValueError('Keine Tournummern gefunden.')

default_tour = all_tours[0]
print('Standardtour:', default_tour)
print('Erste 20 Tournummern:', all_tours[:20])


Standardtour: H/42374
Erste 20 Tournummern: ['H/42374', 'H/42375', 'H/42376', 'H/42377', 'H/42378', 'H/42379', 'H/42380', 'H/42381', 'H/42383', 'H/42391', 'H/42392', 'H/42393', 'H/42395', 'H/42396', 'H/42397', 'H/42398', 'H/42400', 'H/42403', 'H/42404', 'H/42405']


In [ ]:
# Erstelle die Basiskarte mit einem Startzoom und hellen Kartenkacheln
m = folium.Map(
    location=[54.3, 10.1],
    zoom_start=8,
    tiles='CartoDB positron',
    control_scale=True
)

# Dictionary für die Kartenbegrenzungen jeder Tour und Liste für JS-Layer-Namen
bounds_by_tour = {}
layer_js_names = []

for tour in all_tours:
    # Erzeuge eine eigene FeatureGroup für jede Tour
    fg = folium.FeatureGroup(name=f'Tour {tour}', show=(tour == default_tour))
    tour_points = []

    # Filtere Soll- und Ist-Daten nach Tournummer
    soll_t = df_soll[df_soll['TOURNR'] == tour].copy()
    ist_t = df_ist[df_ist['TOURNR'] == tour].copy()

    # Zeichne Soll-Tour und Marker, falls Daten vorhanden sind
    if not soll_t.empty:
        soll_coords = soll_t[['GEOY', 'GEOX']].values.tolist()
        folium.PolyLine(
            locations=soll_coords,
            color='blue',
            weight=4,
            opacity=0.85,
            tooltip=f'Soll-Tour {tour}'
        ).add_to(fg)

        for _, row in soll_t.iterrows():
            lat, lon = row['GEOY'], row['GEOX']
            tour_points.append([lat, lon])

            tooltip_html = f"""
            <div style=\"font-size:13px;\">
                <b>Soll</b><br>
                Tour: {tour}<br>
                Stopp: {int(row['stop_no'])}<br>
                Abfahrt: {fmt_dt(row['ABFAHRT_dt'])}<br>
                Ankunft: {fmt_dt(row['ANKUNFT_dt'])}
            </div>
            """

            folium.Marker(
                [lat, lon],
                tooltip=folium.Tooltip(tooltip_html, sticky=True),
                popup=folium.Popup(tooltip_html, max_width=260),
                icon=BeautifyIcon(
                    icon_shape='marker',
                    number=int(row['stop_no']),
                    border_color='#1d4ed8',
                    text_color='#1d4ed8',
                    background_color='#dbeafe'
                )
            ).add_to(fg)

    # Zeichne Ist-Tour und Marker, falls Daten vorhanden sind
    if not ist_t.empty:
        ist_coords = ist_t[['Breitengrad', 'Längengrad']].values.tolist()
        folium.PolyLine(
            locations=ist_coords,
            color='red',
            weight=4,
            opacity=0.85,
            dash_array='8,6',
            tooltip=f'Ist-Tour {tour}'
        ).add_to(fg)

        for _, row in ist_t.iterrows():
            lat, lon = row['Breitengrad'], row['Längengrad']
            tour_points.append([lat, lon])

            tooltip_html = f"""
            <div style=\"font-size:13px;\">
                <b>Ist</b><br>
                Tour: {tour}<br>
                Stopp: {int(row['stop_no'])}<br>
                Zeitstempel Fahrtzeug: {fmt_dt(row['Meldungszeit'])}<br>
            </div>
            """

            folium.Marker(
                [lat, lon],
                tooltip=folium.Tooltip(tooltip_html, sticky=True),
                popup=folium.Popup(tooltip_html, max_width=260),
                icon=BeautifyIcon(
                    icon_shape='marker',
                    number=int(row['stop_no']),
                    border_color='#b91c1c',
                    text_color='#b91c1c',
                    background_color='#fee2e2'
                )
            ).add_to(fg)

    # Berechne die Bounding-Box der Tour, damit die Karte passend zoomt
    if tour_points:
        lats = [p[0] for p in tour_points]
        lons = [p[1] for p in tour_points]
        bounds_by_tour[tour] = [[min(lats), min(lons)], [max(lats), max(lons)]]

    fg.add_to(m)
    layer_js_names.append((f'Tour {tour}', fg.get_name(), tour))

# Zeige das Layer-Control zur Ein-/Auswahl der Touren an
folium.LayerControl(collapsed=False).add_to(m)

# HTML-Legende für die Karte erzeugen
legend_html = """
<div style="
 position: fixed;
 bottom: 20px;
 left: 20px;
 z-index: 9999;
 background: white;
 border: 1px solid #ccc;
 border-radius: 8px;
 padding: 10px 12px;
 font-size: 13px;
 box-shadow: 0 2px 8px rgba(0,0,0,0.15);
">
  <div style="font-weight:600; margin-bottom:6px;">Legende</div>
  <div><span style="display:inline-block;width:14px;height:4px;background:blue;margin-right:6px;vertical-align:middle;"></span> Soll</div>
  <div><span style="display:inline-block;width:14px;height:4px;background:red;margin-right:6px;vertical-align:middle;"></span> Ist</div>
  <div style="margin-top:6px;color:#555;">Zahl = Reihenfolge des Stopps</div>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# JavaScript zur Steuerung des Kartenzoomes beim Ein- und Ausblenden von Layern
map_var = m.get_name()
layer_map_js = ',\n'.join([f'\"{label}\": {js_name}' for label, js_name, _ in layer_js_names])
bounds_js = ',\n'.join([f'\"{tour}\": {bounds}' for tour, bounds in bounds_by_tour.items()])

script = f"""
{{% macro script(this, kwargs) %}}
var layerMap = {{
{layer_map_js}
}};

var tourBounds = {{
{bounds_js}
}};

var mapObj = {map_var};

function fitTourByLayerName(layerName) {{
    var tour = layerName.replace('Tour ', '');
    var b = tourBounds[tour];
    if (b) {{
        mapObj.fitBounds(b, {{
            padding: [40, 40],
            maxZoom: 15
        }});
    }}
}}

mapObj.whenReady(function() {{
    fitTourByLayerName('Tour {default_tour}');
}});

mapObj.on('overlayadd', function(e) {{
    for (const [name, layer] of Object.entries(layerMap)) {{
        if (e.layer === layer) {{
            fitTourByLayerName(name);
            break;
        }}
    }}
}});
{{% endmacro %}}
"""

macro = MacroElement()
macro._template = Template(script)
m.get_root().add_child(macro)

# Speichere die fertige Karte als HTML-Datei
m.save(OUTPUT_HTML)
print(f'Karte gespeichert als: {OUTPUT_HTML}')


Karte gespeichert als: tourenkarte_emr_alle.html
